In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b_ATS = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv_ATS = Z_b*ω_b_ATS
E_Cb_ATS = C_b_inv_ATS/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=2ω_a# Drive frequency
ω_p=2ω_a-ω_d

ϵ_p = 0.588   # inside the stable period-2 window at ϵ_d = 10.12 g2 (period doubling at 0.578, Neimark-Sacker at 0.59); was 0.635 with the former J1 definition of g2
g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
ϵ_d = 10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b= pi/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#κ_b/2#3e-3*κ_2 #from Leon's paper

# First change of variables
Q_a, Q_b = 1/κ_a, 1/κ_b
Q_a_tilde, Q_b_tilde_ATS = ω_a/κ_a, ω_b_ATS/κ_b
β_a, β_b_ATS= 4E_J*sin(ϵ_p)φ_a^2/(ω_a), 4E_J*sin(ϵ_p)φ_b^2/ω_b_ATS
λ_a, λ_b = φ_a, φ_b
ϵ_λ_d_ATS = ϵ_d*2*λ_b/ω_b_ATS
ν_d = 1;

# Second change of variables
κ_a_tilde_ATS, κ_b_tilde_ATS = 1/(4*Q_a_tilde*sqrt(1-1/(4Q_b_tilde_ATS^2))), 1/(2*Q_b_tilde_ATS*sqrt(1-1/(4Q_b_tilde_ATS^2)))
β_a_tilde_ATS, β_b_tilde_ATS = β_a/(2-1/(2Q_b_tilde_ATS^2)), β_b_ATS/(1-1/(4Q_b_tilde_ATS^2))
ϵ_d_tilde_ATS = ϵ_λ_d_ATS/(1-1/(4Q_b_tilde_ATS^2))
ν_d_tilde_ATS = ν_d/sqrt(1-1/(4Q_b_tilde_ATS^2))
γ_ATS = (1-1/(4Q_a_tilde^2))/(1-1/(4Q_b_tilde_ATS^2))/2;

In [ ]:
t_max = 100/κ_b
t_min = 0.0
t_min_save = t_max - 2pi/(ω_a)*2
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(30sqrt(2)*3ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)


drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-6,6,10)
Re_a_list=range(-6,6,10)

Final_mode_a_ATS = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b_ATS = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_a_ATS_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b_ATS_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = re_i + 1im*im_j

        system = SystemATS_ladder(ω_a, ω_b_ATS, E_J, E_Ca, E_Cb_ATS, a_0_j, b_0)
        system_prepared = prepare(system)
        environment_prepared = prepare(system,environment);

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)
        x_a, p_a = (a+conj(a))/sqrt(2), 1im*(conj(a)-a)/sqrt(2)
        x_b, p_b = (b+conj(b))/sqrt(2), 1im*(conj(b)-b)/sqrt(2)

        x_lam_a, x_lam_b, p_lam_a, p_lam_b = sqrt(2)*x_a, sqrt(2)*x_b, sqrt(2)*p_a, sqrt(2)*p_b
        u_a =@. cos(ω_b_ATS*t_list/2)*x_lam_a - sin(ω_b_ATS*t_list/2)*p_lam_a
        v_a =@. sin(ω_b_ATS*t_list/2)*x_lam_a + cos(ω_b_ATS*t_list/2)*p_lam_a
        u_b =@. cos(ω_b_ATS*t_list)*x_lam_b- sin(ω_b_ATS*t_list)*p_lam_b
        v_b =@. sin(ω_b_ATS*t_list)*x_lam_b + cos(ω_b_ATS*t_list)*p_lam_b;

        a_R = (u_a+1im*v_a)/2*1im # = i a e^{iω_a t}, in units of a (u+iv = 2a e^{iωt}; was /sqrt(2), i.e. √2 a, until 12/10/2026)
        b_R = (u_b+1im*v_b)/2*1im #Rotation 90°
        if maximum(abs.(imag(a_R))) > 1.0   # satellites sit at |Im a| ≈ 4.2, cats on the real axis (was > 0.2, too tight for the larger cats)
            Final_mode_a_ATS[(re_i,im_j)] = a_R
            Final_mode_b_ATS[(re_i,im_j)] = b_R
        else 
            Final_mode_a_ATS_cat[(re_i,im_j)] = a_R
            Final_mode_b_ATS_cat[(re_i,im_j)] = b_R
        end    
    end
end

In [ ]:
p0 = plot()
for key in keys(Final_mode_a_ATS)
    z = Final_mode_a_ATS[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_a_ATS_cat)
    z = Final_mode_a_ATS_cat[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end


plot!(p0, xlabel = L"\mathrm{Re}(a)", ylabel = L"\mathrm{Im}(a)", aspect_ratio = 1, guidefontsize=12)


p1 = plot()
for key in keys(Final_mode_b_ATS)
    z = Final_mode_b_ATS[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_b_ATS_cat)
    z = Final_mode_b_ATS_cat[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end

plot!(p1, xlabel = L"\mathrm{Re}(b)", ylabel = L"\mathrm{Im}(b)", aspect_ratio = 1, guidefontsize=12)
plot(p0, p1, layout = (1, 2))
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/limit_cycles/classical_limit_cycles_period_doubled_12_10_2026.pdf")